### RAG pipeline
Set up prompt and RAG chain, test queries.

In [0]:
%pip install -r requirements.txt
dbutils.library.restartPython()

In [0]:
INDEX_PATH = "faiss_index" 
MODEL_NAME = "all-MiniLM-L6-v2"
GROQ_MODEL = "groq/compound-mini"

In [0]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)
vector_store = FAISS.load_local(
    INDEX_PATH, 
    embeddings, 
    allow_dangerous_deserialization=True
)

In [0]:
import os
from getpass import getpass
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [0]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model_name="groq/compound-mini", temperature=0)

# prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a professional UK Financial Compliance Assistant. 
Answer the user's question using ONLY the provided context from the FCA COBS handbook (Chapters 1-10A).
    
CRITICAL INSTRUCTIONS:
1. Do NOT start your answer with phrases like "Based on the context," "Based on the provided text," or "According to the documents."
2. Start your answer **directly** with the factual information.
3. If the answer is not in the context, simply state: "I cannot find this information in the provided COBS chapters."
4. Do not hallucinate. Be precise and professional.

Context: {context}
Question: {question}

Helpful answer:"""),
    ("human", "{question}")
])

# build LCEL Chain
# format prompt -> pass to LLM -> parse output as string
retriever = vector_store.as_retriever(search_kwargs={"k": 6})

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt 
    | llm 
    | StrOutputParser()
)

In [0]:
import time
import random

def delay_query():
    time.sleep(random.uniform(15, 30))

In [0]:
delay_query()

query = "What are the requirements for client categorisation?"
response = rag_chain.invoke(query)

print(response)

In [0]:
delay_query()

query = "What are the rules on inducements?"
response = rag_chain.invoke(query)

print(response)

In [0]:

time.sleep(random.uniform(5, 15))
query = "Which potential clients might need enhanced KYC?" # this one should return a cannot find
response = rag_chain.invoke(query)

print(response)

In [0]:
delay_query()
query = "What is the first line of COBS 1?" # quick test
response = rag_chain.invoke(query)

print(response)